In [ ]:
# 1.遍历
#   raw / case_xxx / (2postnatal, 3postoperative) / visit_id / meta.json
# 2.对这些 meta.json：
#   如果存在 "gestational_age_days"
#   → 改成 "postnatal_age_days"
#   其他字段保持不变
# 3.1prenatal 不处理



from pathlib import Path
import json

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

updated_count = 0
skipped_count = 0
error_count = 0

# 只处理这两个阶段
target_stages = {"2postnatal", "3postoperative"}

for case_dir in root_dir.iterdir():
    if not case_dir.is_dir():
        continue

    for stage_dir in case_dir.iterdir():
        if not stage_dir.is_dir():
            continue

        # 只处理 2postnatal 和 3postoperative
        if stage_dir.name not in target_stages:
            continue

        for visit_dir in stage_dir.iterdir():
            if not visit_dir.is_dir():
                continue

            meta_path = visit_dir / "meta.json"

            if not meta_path.exists():
                print(f"[跳过] 不存在 meta.json: {meta_path}")
                skipped_count += 1
                continue

            try:
                content = meta_path.read_text(encoding="utf-8").strip()

                if content == "":
                    print(f"[跳过] 空文件: {meta_path}")
                    skipped_count += 1
                    continue

                data = json.loads(content)

                # 如果已经改过了，就跳过
                if "postnatal_age_days" in data:
                    print(f"[跳过] 已是 postnatal: {meta_path}")
                    skipped_count += 1
                    continue

                # 如果有 gestational_age_days，则改名
                if "gestational_age_days" in data:
                    value = data.pop("gestational_age_days")
                    data["postnatal_age_days"] = value

                    # 写回文件
                    with open(meta_path, "w", encoding="utf-8") as f:
                        json.dump(data, f, ensure_ascii=False, indent=2)

                    print(f"[已修改] {meta_path}")
                    updated_count += 1
                else:
                    print(f"[跳过] 无该字段: {meta_path}")
                    skipped_count += 1

            except Exception as e:
                print(f"[错误] {meta_path}: {e}")
                error_count += 1

print("\n处理完成")
print(f"修改数量: {updated_count}")
print(f"跳过数量: {skipped_count}")
print(f"错误数量: {error_count}")